# SAR Pipeline Validation Notebook
## Single Acquisition — End-to-End Processing Check

**Purpose:** Download one Sentinel-1 GRD scene, run every preprocessing step (S0→S9),
validate each output, and export QGIS-ready files.

Run this notebook BEFORE `nb_sar_09_accra_flood_intelligence.ipynb` to confirm:
- Copernicus credentials work
- pygeofetch downloads and reprojects correctly
- GCP recovery fixes the identity-transform bug
- Speckle filter, dB conversion, normalisation all produce valid outputs
- QGIS can render all output files without manual stretching

**Study area:** Odaw Basin, Accra, Ghana (EPSG:32630 / UTM Zone 30N)


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  SAR Pipeline Validation — Cell 1: Imports & Setup      ║
# ╚══════════════════════════════════════════════════════════╝
import warnings; warnings.filterwarnings("ignore")
import pathlib
import shutil
import subprocess
import time

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import rasterio

# ── PyGeoVision core ──────────────────────────────────────────────────────────
import pygeovision as pgv
from pygeovision import PyGeoVision

# ── SAR bug-fix modules (Jul 2026) ───────────────────────────────────────────
from pygeovision.data.processors.sar import (
    clip_sar_to_bbox,
    despeckle_sar,
    linear_to_db,
    normalise_sar_for_ai,
    validate_sar_georeference,
)
from pygeovision.data.validators.georeference import (
    check_download_complete,
    reproject_bbox_to_raster_crs,
)

client = PyGeoVision()

print(f"pygeovision  : {pgv.__version__}")
print(f"rasterio     : {rasterio.__version__}")
print()
print("All imports OK")


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
#
# Edit COPERNICUS_USER and COPERNICUS_PASS before running.
# The bbox uses a small test area (Korle Lagoon sub-basin ~50 km²)
# so the download is fast (~400 MB instead of 1.5 GB full scene).

COPERNICUS_USER = "appiahkubis14@gmail.com"
COPERNICUS_PASS = "Copernicus@esa@2001server"

# Small test bbox — covers Korle Lagoon + Agbogbloshie (flood-prone, fast download)
TEST_BBOX    = (-0.25, 5.52, -0.10, 5.62)   # (lon_min, lat_min, lon_max, lat_max)
TARGET_CRS   = "EPSG:32630"                  # UTM Zone 30N, correct for Accra
S1_BANDS     = ["vv", "vh"]

# Search a recent dry-season window (clear skies = good baseline)
SEARCH_DATE_RANGE = ("2025-01-01", "2025-02-28")

# Output directories
VAL_DIR   = pathlib.Path("./sar_validation")
RAW_DIR   = VAL_DIR / "01_raw"
WORK_DIR  = VAL_DIR / "02_work"
QGIS_DIR  = VAL_DIR / "03_qgis"
for d in [VAL_DIR, RAW_DIR, WORK_DIR, QGIS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Add credentials
client.add_credentials("copernicus",
                        username=COPERNICUS_USER,
                        password=COPERNICUS_PASS)

print("Configuration:")
print(f"  Test bbox     : {TEST_BBOX}")
print(f"  Target CRS    : {TARGET_CRS}")
print(f"  Search window : {SEARCH_DATE_RANGE}")
print(f"  Output dir    : {VAL_DIR.absolute()}")
print()
print("Credentials registered. Ready to search.")


In [ ]:
# ── Step 1: Search for Sentinel-1 GRD scenes ────────────────────────────────
print("Searching Copernicus for Sentinel-1 GRD scenes...")
print(f"  Area  : {TEST_BBOX}")
print(f"  Period: {SEARCH_DATE_RANGE[0]} → {SEARCH_DATE_RANGE[1]}")
print()

scenes = client.search(
    bbox           = TEST_BBOX,
    date_range     = SEARCH_DATE_RANGE,
    satellite      = "Sentinel-1",
    providers      = ["planetary_computer"],
    collections    = ["sentinel-1-grd"],
    cloud_cover_max= 100,   # SAR is cloud-independent
    use_cache      = False,
)

if not scenes:
    print("ERROR: No scenes found. Check credentials and date range.")
else:
    print(f"Found {len(scenes)} scenes:")
    for i, s in enumerate(scenes[:5]):
        dt   = getattr(s, "datetime", None) or getattr(s, "date", "unknown")
        pid  = getattr(s, "product_id", getattr(s, "id", "?"))
        orb  = s.extra.get("relative_orbit", "?") if hasattr(s, "extra") else "?"
        pas  = s.extra.get("pass_direction", "?") if hasattr(s, "extra") else "?"
        print(f"  [{i}] {dt}  orbit={orb}  pass={pas}  id={str(pid)[:40]}")
    if len(scenes) > 5:
        print(f"  ... and {len(scenes)-5} more")
    print()
    # Pick the most recent scene
    SELECTED = scenes[0]
    dt_sel   = getattr(SELECTED, "datetime", "?")
    print(f"Selected scene [0]: {dt_sel}")


In [ ]:
# ── Step 2: Download the selected scene ──────────────────────────────────────
# post_process=['reproject:EPSG:32630', 'cog'] triggers the pygeofetch step
# that is KNOWN to produce the identity-transform CRS corruption bug.
# This is intentional — we want to reproduce the bug, then fix it.

print("Downloading Sentinel-1 GRD scene...")
print(f"  post_process: ['reproject:{TARGET_CRS}', 'cog']")
print(f"  Output dir  : {RAW_DIR}")
print()
t0 = time.time()

dl_results = client.download(
    [SELECTED],
    output_dir   = str(RAW_DIR),
    bands        = S1_BANDS,
    post_process = [f"reproject:{TARGET_CRS}", "cog"],
)

elapsed = time.time() - t0
print(f"Download completed in {elapsed:.0f}s")
print()

if not dl_results or not dl_results[0].success:
    err = dl_results[0].error if dl_results else "no result"
    print(f"ERROR: Download failed — {err}")
    RAW_PATH = None
else:
    RAW_PATH = pathlib.Path(dl_results[0].path)
    size_mb  = RAW_PATH.stat().st_size / (1024*1024) if RAW_PATH.exists() else 0
    print("Downloaded file:")
    print(f"  Path    : {RAW_PATH}")
    print(f"  Size    : {size_mb:.1f} MB")
    print()
    # Quick rasterio check
    with rasterio.open(str(RAW_PATH)) as src:
        print(f"  Bands   : {src.count}")
        print(f"  Shape   : {src.height} × {src.width} px")
        print(f"  CRS tag : {src.crs}")
        print(f"  Transform a (pixel width): {src.transform.a:.4f}")
        print(f"  Origin  : ({src.transform.c:.1f}, {src.transform.f:.1f})")
        print()
        if abs(src.transform.a - 1.0) < 0.01:
            print("  ⚠  IDENTITY TRANSFORM DETECTED (a≈1.0)")
            print("  This is the known pygeofetch CRS corruption bug.")
            print("  Cell 5 (S1 validation + GCP recovery) will fix it.")
        else:
            print(f"  ✓ Transform looks valid (pixel size ≈{src.transform.a:.1f} m)")


In [ ]:
# ── S0: Download completeness verification ────────────────────────────────────
print("S0 — Verifying download completeness...")
print()

if RAW_PATH is None:
    print("SKIP: no raw file")
else:
    dlc = check_download_complete(str(RAW_PATH))

    status = "✓ COMPLETE" if dlc["complete"] else "✗ INCOMPLETE"
    print(f"  Status         : {status}")
    # print(f"  File size      : {dlc['file_size_mb']:.1f} MB")
    print(f"  Readable tiles : {dlc['readable_tiles']} / {dlc['total_tiles']}")

    if dlc["errors"]:
        for e in dlc["errors"]:
            print(f"  ERROR: {e}")

    if not dlc["complete"]:
        print()
        print("  Action: re-run the download cell or check disk space.")
    else:
        print()
        print("  All tiles readable — proceed to S1.")


In [ ]:
# ── S1: Georeference validation + GCP recovery ───────────────────────────────
# This is the critical fix for the identity-transform corruption bug.
# validate_sar_georeference() detects the corrupt transform, reads the
# embedded GCPs (210-231 per file), calls rasterio.transform.from_gcps(),
# and writes a repaired copy to _repaired/ alongside the original.

print("S1 — Georeference validation + GCP recovery...")
print()

if RAW_PATH is None:
    print("SKIP: no raw file")
    geo = None
    WORKING_PATH = None
else:
    geo = validate_sar_georeference(str(RAW_PATH))

    print(f"  valid        : {geo.valid}")
    # print(f"  gcp_count    : {geo.gcp_count}")
    print(f"  errors       : {geo.errors}")
    print(f"  warnings     : {geo.warnings}")
    # print(f"  recovered    : {geo.recovered}")
    print(f"  repaired_path: {geo.repaired_path}")
    print()

    if geo.valid and not geo.recovered:
        print("  ✓ Georeference valid — no repair needed.")
        WORKING_PATH = str(RAW_PATH)

    elif geo.repaired_path:
        print("  ✓ GCP recovery succeeded.")
        WORKING_PATH = geo.repaired_path

        # Verify the repaired file
        with rasterio.open(WORKING_PATH) as src:
            t = src.transform
            print()
            print("  Repaired file verification:")
            print(f"    CRS     : {src.crs}")
            print(f"    Pixel a : {t.a:.2f} m  (expected ~10.0 for S-1 GRD)")
            print(f"    Origin  : ({t.c:.0f}E, {t.f:.0f}N) UTM")
            print(f"    Shape   : {src.height} × {src.width} px")
            valid_pixel = abs(t.a - 10.0) < 5.0  # within 5 m of expected
            if valid_pixel:
                print("    ✓ Pixel size physically sensible")
            else:
                print(f"    ✗ Pixel size {t.a:.2f} m unexpected — check manually")
    else:
        print("  ✗ Georeference invalid and unrecoverable.")
        print("  Cannot proceed. Try re-downloading the file.")
        WORKING_PATH = None


In [ ]:
# ── S6: Speckle filtering (Enhanced Lee, on LINEAR data) ─────────────────────
# CRITICAL ORDER RULE: despeckle BEFORE dB conversion.
# The Enhanced Lee filter assumes multiplicative Gamma noise — valid only
# for linear power values. Filtering dB-scale data produces biased results.

print("S6 — Speckle filtering (Enhanced Lee, window=7, num_looks=4)...")
print()

if WORKING_PATH is None:
    print("SKIP: no valid working file")
    DESP_PATH = None
else:
    stem      = pathlib.Path(WORKING_PATH).stem
    DESP_PATH = str(WORK_DIR / f"{stem}_S6_despeckle.tif")

    t0 = time.time()
    despeckle_sar(
        WORKING_PATH, DESP_PATH,
        filter_type  = "enhanced_lee",
        window_size  = 7,
        num_looks    = 4,
    )
    elapsed = time.time() - t0

    with rasterio.open(DESP_PATH) as src:
        data = src.read().astype("float32")

    print(f"  Output       : {pathlib.Path(DESP_PATH).name}")
    print(f"  Time         : {elapsed:.1f}s")
    print(f"  Shape        : {data.shape}")
    for b in range(data.shape[0]):
        band = data[b]
        valid = band[band > 0]
        if len(valid):
            print(f"  Band {b+1} range : [{valid.min():.6f}, {valid.max():.6f}]"
                  f"  (linear power — should be positive small numbers)")

    # Verify: all values should be positive (linear power > 0)
    if (data > 0).all():
        print("  ✓ All values positive (correct linear power domain)")
    else:
        neg_pct = (data <= 0).mean() * 100
        print(f"  ⚠ {neg_pct:.1f}% non-positive pixels (check for nodata handling)")


In [ ]:
# ── S7: dB conversion (AFTER despeckle) ──────────────────────────────────────
# Formula: dB = 10 × log10(max(linear, 1e-10))
# Clipped to [-35, +5] dB — the physical SAR dynamic range for Sentinel-1.
# Physical interpretation:
#   < -25 dB: open water / calm sea
#   -18 to -22 dB: potential flood inundation
#   -12 to -16 dB: tropical vegetation
#   -5 to +2 dB: urban double-bounce

print("S7 — dB conversion (clip: -35 to +5 dB)...")
print()

if DESP_PATH is None:
    print("SKIP")
    DB_PATH = None
else:
    stem    = pathlib.Path(DESP_PATH).stem
    DB_PATH = str(WORK_DIR / f"{stem.replace('_S6_despeckle','')}_S7_db.tif")

    t0 = time.time()
    linear_to_db(DESP_PATH, DB_PATH, clip_db_min=-35.0, clip_db_max=5.0)
    elapsed = time.time() - t0

    with rasterio.open(DB_PATH) as src:
        data = src.read().astype("float32")

    print(f"  Output : {pathlib.Path(DB_PATH).name}")
    print(f"  Time   : {elapsed:.1f}s")
    print()
    print("  Band statistics (dB):")
    band_names = {1: "VV", 2: "VH"}
    for b in range(data.shape[0]):
        band = data[b]
        valid = band[(band > -35) & (band < 6)]
        if len(valid):
            pct_flood = (valid < -18).mean() * 100
            print(f"    {band_names.get(b+1, f'Band {b+1}')}: "
                  f"min={valid.min():.1f}  max={valid.max():.1f}  "
                  f"mean={valid.mean():.1f}  std={valid.std():.1f} dB")
            if b == 1:  # VH
                print(f"    VH pixels below -18 dB flood threshold: {pct_flood:.1f}%")

    print()
    # VH mean for Accra dry season should be around -15 to -20 dB
    if data.shape[0] >= 2:
        vh = data[1]
        vh_mean = float(vh[(vh > -35) & (vh < 6)].mean())
        if -25 < vh_mean < -8:
            print(f"  ✓ VH mean {vh_mean:.1f} dB is physically plausible for Accra")
        else:
            print(f"  ⚠ VH mean {vh_mean:.1f} dB outside expected range [-25, -8] dB")


In [ ]:
# ── S8: Normalisation for AI [0, 1] ──────────────────────────────────────────
# Formula: normalised = (dB - DB_MIN) / (DB_MAX - DB_MIN)
#          normalised = (dB + 35) / 40    for range [-35, +5]
# Fixed physical bounds ensure cross-scene comparability.
# The same VH value (-18 dB → 0.425) means the same thing in every scene.

print("S8 — Normalisation to [0, 1] (physical min-max)...")
print()

if DB_PATH is None:
    print("SKIP")
    NORM_PATH = None
else:
    stem      = pathlib.Path(DB_PATH).stem
    NORM_PATH = str(WORK_DIR / f"{stem.replace('_S7_db','')}_S8_norm.tif")

    t0 = time.time()
    normalise_sar_for_ai(DB_PATH, NORM_PATH, method="minmax_db",
                         db_min=-35.0, db_max=5.0)
    elapsed = time.time() - t0

    with rasterio.open(NORM_PATH) as src:
        data = src.read().astype("float32")
        meta = src.meta

    print(f"  Output : {pathlib.Path(NORM_PATH).name}")
    print(f"  Time   : {elapsed:.1f}s")
    print(f"  CRS    : {meta['crs']}")
    print()
    print("  Band statistics (normalised [0,1]):")
    band_names = {1: "VV", 2: "VH"}
    for b in range(data.shape[0]):
        band = data[b].ravel()
        valid = band[np.isfinite(band)]
        if len(valid):
            flood_thresh_norm = (-18.0 + 35.0) / 40.0   # = 0.425
            pct_flood = (valid < flood_thresh_norm).mean() * 100
            print(f"    {band_names.get(b+1, f'Band {b+1}')}: "
                  f"min={valid.min():.4f}  max={valid.max():.4f}  "
                  f"mean={valid.mean():.4f}")
            if b == 1:
                print(f"    VH pixels below flood threshold (0.425 ≡ -18 dB): {pct_flood:.1f}%")

    # Verify: all values should be in [0, 1]
    total = data.size
    in_range = ((data >= 0) & (data <= 1)).sum()
    print()
    print(f"  ✓ {in_range/total*100:.2f}% of pixels in valid [0,1] range")
    if in_range < total:
        print(f"  ⚠ {total-in_range} pixels outside [0,1] — check normalisation")


In [ ]:
# ── S9: Clip to study area (CRS-aware) ───────────────────────────────────────
# BUG 3 fix: clip_sar_to_bbox() reprojects the WGS84 bbox to the raster CRS
# before clipping. Without this, the UTM raster and WGS84 bbox don't intersect.

print("S9 — Clip to test bbox (CRS-aware reprojection)...")
print()
print(f"  Input bbox (WGS84) : {TEST_BBOX}")

if NORM_PATH is None:
    print("SKIP")
    CLIP_PATH = None
else:
    # Show what the bbox becomes in UTM
    from pygeovision.data.validators.georeference import reproject_bbox_to_raster_crs
    utm_bbox = reproject_bbox_to_raster_crs(TEST_BBOX, NORM_PATH)
    print(f"  Reprojected (UTM)  : {tuple(f'{v:.0f}' for v in utm_bbox)}")
    print()

    stem      = pathlib.Path(NORM_PATH).stem
    CLIP_PATH = str(WORK_DIR / f"{stem.replace('_S8_norm','')}_S9_clip.tif")

    t0 = time.time()
    try:
        clip_sar_to_bbox(NORM_PATH, CLIP_PATH, TEST_BBOX)
        elapsed = time.time() - t0

        with rasterio.open(CLIP_PATH) as src:
            data   = src.read().astype("float32")
            bounds = src.bounds
            crs    = src.crs
            t      = src.transform

        print(f"  Output    : {pathlib.Path(CLIP_PATH).name}")
        print(f"  Time      : {elapsed:.1f}s")
        print(f"  Shape     : {data.shape}")
        print(f"  CRS       : {crs}")
        print(f"  Bounds    : {bounds}")
        print(f"  Pixel size: {t.a:.2f} m")
        print()

        # Verify: data should not be all zeros or all ones
        vh = data[1] if data.shape[0] >= 2 else data[0]
        valid = vh[np.isfinite(vh)]
        if valid.std() > 0.01:
            print(f"  ✓ Clipped data has variance (std={valid.std():.4f}) — not blank")
        else:
            print("  ✗ Clipped data has near-zero variance — may be misclipped")

    except ValueError as e:
        print(f"  ✗ Clip failed: {e}")
        print("  Falling back to full-extent output")
        shutil.copy2(NORM_PATH, CLIP_PATH)
        CLIP_PATH = NORM_PATH


In [ ]:
# ── QGIS Export: embed statistics + overviews + QML style ───────────────────
# This cell produces three QGIS-ready files that render WITHOUT manual stretching:
#   1. float32 normalised [0,1] with embedded stats → _norm_qgis.tif
#   2. float32 dB scale [-35, +5] with pseudocolour QML → _dB_qgis.tif
#   3. uint8 [0,255] guaranteed-render fallback → _uint8.tif

print("QGIS Export...")
print()

def embed_stats_overviews(src_path, dst_path):
    """Copy file, embed statistics, build overview pyramids."""
    try:
        result = subprocess.run(
            ["gdal_translate", "-stats",
             "-co", "TILED=YES", "-co", "COMPRESS=LZW",
             str(src_path), str(dst_path)],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            shutil.copy2(str(src_path), str(dst_path))
    except FileNotFoundError:
        shutil.copy2(str(src_path), str(dst_path))

    try:
        subprocess.run(
            ["gdaladdo", "-r", "average", str(dst_path), "2", "4", "8", "16"],
            capture_output=True, check=True
        )
        return True
    except Exception:
        return False


def write_qml_grey(path, b_min, b_max, band=2):
    qml = f"""<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.34"><pipe>
<provider><resampling enabled="false" zoomedInResamplingMethod="nearestNeighbour"
 zoomedOutResamplingMethod="nearestNeighbour" maxOversampling="2"/></provider>
<rasterrenderer opacity="1" alphaBand="-1" band="{band}" type="singlebandgray">
<rasterTransparency/>
<contrastEnhancement>
  <minValue>{b_min:.6f}</minValue><maxValue>{b_max:.6f}</maxValue>
  <algorithm>StretchToMinimumMaximum</algorithm>
</contrastEnhancement>
<colorGradient>BlackToWhite</colorGradient>
</rasterrenderer>
<brightnesscontrast brightness="0" contrast="0" gamma="1"/>
<rasterresampler maxOversampling="2"/></pipe></qgis>"""
    with open(path, "w") as f: f.write(qml)


def write_qml_db(path, band=2):
    qml = """<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.34"><pipe>
<provider><resampling enabled="false" zoomedInResamplingMethod="nearestNeighbour"
 zoomedOutResamplingMethod="nearestNeighbour" maxOversampling="2"/></provider>
<rasterrenderer opacity="1" alphaBand="-1" band="2" type="singlebandpseudocolor">
<rasterTransparency/>
<rastershader>
<colorrampshader colorRampType="INTERPOLATED" minimumValue="-30" maximumValue="0">
<item value="-30" color="#0571b0" alpha="255" label="-30 dB (water/flood)"/>
<item value="-20" color="#92c5de" alpha="255" label="-20 dB"/>
<item value="-15" color="#f7f7f7" alpha="255" label="-15 dB (vegetation)"/>
<item value="-8"  color="#f4a582" alpha="255" label="-8 dB"/>
<item value="0"   color="#ca0020" alpha="255" label="0 dB (urban)"/>
</colorrampshader>
</rastershader>
</rasterrenderer>
<brightnesscontrast brightness="0" contrast="0" gamma="1"/>
<rasterresampler maxOversampling="2"/></pipe></qgis>"""
    with open(path, "w") as f: f.write(qml)


if CLIP_PATH is None:
    print("SKIP: no clipped file")
else:
    # 1. Normalised [0,1] float32 with embedded stats
    norm_qgis = str(QGIS_DIR / "s1_norm_qgis.tif")
    ok = embed_stats_overviews(CLIP_PATH, norm_qgis)
    with rasterio.open(CLIP_PATH) as src:
        d = src.read().astype("float32")
    vh = d[1] if d.shape[0] >= 2 else d[0]
    valid = vh[np.isfinite(vh)]
    mn = float(np.percentile(valid, 2))
    mx = float(np.percentile(valid, 98))
    write_qml_grey(norm_qgis.replace(".tif", ".qml"), mn, mx)
    ovr = "✓ overviews" if ok else "no overviews (gdaladdo not found)"
    print(f"  1. Normalised float32 : s1_norm_qgis.tif  ({ovr})")
    print(f"     VH display range   : [{mn:.4f}, {mx:.4f}]  (2nd–98th pct)")

    # 2. dB scale from work dir
    if DB_PATH:
        db_qgis = str(QGIS_DIR / "s1_dB_qgis.tif")
        try:
            # Clip the dB file to the same bbox
            db_clip_tmp = str(WORK_DIR / "_db_clip_tmp.tif")
            try:
                clip_sar_to_bbox(DB_PATH, db_clip_tmp, TEST_BBOX)
                embed_stats_overviews(db_clip_tmp, db_qgis)
            except Exception:
                embed_stats_overviews(DB_PATH, db_qgis)
            write_qml_db(db_qgis.replace(".tif", ".qml"))
            print("  2. dB pseudocolour    : s1_dB_qgis.tif")
            print("     QML: blue=-30dB (water) → red=0dB (urban)")
        except Exception as e:
            print(f"  2. dB export skipped: {e}")

    # 3. uint8 guaranteed render
    uint8_path = str(QGIS_DIR / "s1_uint8.tif")
    d_u8 = (np.clip(d, 0, 1) * 255).astype("uint8")
    with rasterio.open(CLIP_PATH) as src:
        meta = src.meta.copy()
    meta.update(dtype="uint8", nodata=0)
    with rasterio.open(uint8_path, "w", **meta) as dst:
        dst.write(d_u8)
    try:
        subprocess.run(["gdaladdo", "-r", "average", uint8_path, "2", "4", "8", "16"],
                       capture_output=True, check=True)
    except Exception: pass
    vh_u8 = d_u8[1] if d_u8.shape[0] >= 2 else d_u8[0]
    valid_u8 = vh_u8[vh_u8 > 0]
    mn_u8 = int(np.percentile(valid_u8, 2)) if len(valid_u8) else 0
    mx_u8 = int(np.percentile(valid_u8, 98)) if len(valid_u8) else 255
    write_qml_grey(uint8_path.replace(".tif", ".qml"), mn_u8, mx_u8, band=2)
    print("  3. uint8 (guaranteed) : s1_uint8.tif")
    print(f"     VH display range   : [{mn_u8}, {mx_u8}]")
    print()
    print(f"All QGIS exports in: {QGIS_DIR.absolute()}")


In [ ]:
# ── Visualisation: 6-panel validation plot ───────────────────────────────────
# Shows: raw → repaired → despeckled → dB → normalised → clipped
# Each panel includes the key diagnostic value for that step.

print("Building 6-panel validation plot...")
print()

files = {
    "Raw download\n(may have identity transform)": RAW_PATH,
    "S1: GCP-repaired\n(valid UTM georeference)":  geo.repaired_path if geo and geo.repaired_path else RAW_PATH,
    "S6: Despeckled\n(linear power, Enhanced Lee)": DESP_PATH,
    "S7: dB converted\n(−35 to +5 dB)":             DB_PATH,
    "S8: Normalised\n([0, 1], physical bounds)":    NORM_PATH,
    "S9: Clipped\n(study area, CRS-aware)":         CLIP_PATH,
}

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor("#0D1B2A")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.15)

STEP_COLORS = ["#EF5350","#42A5F5","#26A69A","#FFA726","#66BB6A","#AB47BC"]

for idx, (title, fpath) in enumerate(files.items()):
    ax = fig.add_subplot(gs[idx // 3, idx % 3])
    ax.set_facecolor("#1C2C3C")

    if fpath is None or not pathlib.Path(str(fpath)).exists():
        ax.text(0.5, 0.5, "No file\n(step skipped)",
                ha="center", va="center", color="#90A4AE", fontsize=11,
                transform=ax.transAxes)
        ax.set_title(title, color=STEP_COLORS[idx], fontsize=9, fontweight="bold", pad=6)
        ax.axis("off")
        continue

    try:
        with rasterio.open(str(fpath)) as src:
            # Use VH if available (band 2), else VV
            band_idx = 2 if src.count >= 2 else 1
            data = src.read(band_idx).astype("float32")
            t    = src.transform
            crs  = src.crs

        # Replace 0/NaN with nan for display
        data = np.where((data == 0) | ~np.isfinite(data), np.nan, data)
        valid = data[np.isfinite(data)]

        if len(valid) == 0:
            ax.text(0.5, 0.5, "All pixels\nare NaN/zero",
                    ha="center", va="center", color="#EF9A9A", fontsize=11,
                    transform=ax.transAxes)
        else:
            vmin = np.percentile(valid, 2)
            vmax = np.percentile(valid, 98)
            im = ax.imshow(data, cmap="gray", vmin=vmin, vmax=vmax, aspect="auto")
            plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02,
                        label="Value").ax.yaxis.set_tick_params(color="white", labelcolor="white")

            # Diagnostic text
            diag_lines = [
                f"Shape: {data.shape[0]}×{data.shape[1]}",
                f"Range: [{valid.min():.3f}, {valid.max():.3f}]",
                f"Mean: {valid.mean():.3f}",
                f"Pixel: {t.a:.2f} m",
            ]
            diag = "\n".join(diag_lines)
            ax.text(0.02, 0.02, diag, transform=ax.transAxes,
                    fontsize=7, color="#B0BEC5", va="bottom",
                    bbox=dict(facecolor="#0D1B2A", alpha=0.7, pad=2, edgecolor="none"))

    except Exception as e:
        ax.text(0.5, 0.5, f"Error reading file:\n{str(e)[:40]}",
                ha="center", va="center", color="#EF9A9A", fontsize=9,
                transform=ax.transAxes)

    ax.set_title(title, color=STEP_COLORS[idx], fontsize=9, fontweight="bold", pad=6)
    ax.tick_params(colors="#546E7A", labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(STEP_COLORS[idx])
        spine.set_linewidth(1.5)

fig.suptitle("SAR Pipeline Validation — S0 to S9\nSentinel-1 GRD · Odaw Basin, Accra · VH polarisation",
             color="white", fontsize=13, fontweight="bold", y=1.01)

plt.savefig(str(VAL_DIR / "pipeline_validation.png"),
            dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Plot saved: {VAL_DIR / 'pipeline_validation.png'}")


In [ ]:
# ── Final Validation Report ───────────────────────────────────────────────────
print()
print("=" * 60)
print("SAR PIPELINE VALIDATION REPORT")
print("=" * 60)
print()

checks = {}

# S0: download
if RAW_PATH and RAW_PATH.exists():
    dlc = check_download_complete(str(RAW_PATH))
    checks["S0 Download"] = ("PASS" if dlc["complete"] else "FAIL",
                              f"{dlc['file_size_mb']:.1f} MB, "
                              f"{dlc['readable_tiles']}/{dlc['total_tiles']} tiles")
else:
    checks["S0 Download"] = ("FAIL", "File not found")

# S1: georeference
if geo:
    if geo.recovered:
        checks["S1 Georeference"] = ("PASS (recovered)",
                                      f"GCP recovery from {geo.gcp_count} GCPs")
    elif geo.valid:
        checks["S1 Georeference"] = ("PASS", "Valid transform on first read")
    else:
        checks["S1 Georeference"] = ("FAIL", f"Errors: {geo.errors}")
else:
    checks["S1 Georeference"] = ("SKIP", "No raw file")

# S6: despeckle
if DESP_PATH and pathlib.Path(DESP_PATH).exists():
    with rasterio.open(DESP_PATH) as src:
        d = src.read().astype("float32")
    ok = (d > 0).mean() > 0.95
    checks["S6 Despeckle"] = ("PASS" if ok else "WARN",
                               f"Linear power, {d.shape}, {(d>0).mean()*100:.0f}% positive")
else:
    checks["S6 Despeckle"] = ("SKIP", "No despeckle file")

# S7: dB
if DB_PATH and pathlib.Path(DB_PATH).exists():
    with rasterio.open(DB_PATH) as src:
        d = src.read().astype("float32")
    vh = d[1] if d.shape[0] >= 2 else d[0]
    valid = vh[(vh > -35) & (vh < 6)]
    ok = len(valid) > 0 and -25 < valid.mean() < -8
    checks["S7 dB Convert"] = ("PASS" if ok else "WARN",
                                f"VH mean={valid.mean():.1f} dB (expected -25 to -8)")
else:
    checks["S7 dB Convert"] = ("SKIP", "No dB file")

# S8: normalise
if NORM_PATH and pathlib.Path(NORM_PATH).exists():
    with rasterio.open(NORM_PATH) as src:
        d = src.read().astype("float32")
    in_range = ((d >= 0) & (d <= 1)).mean()
    ok = in_range > 0.99
    checks["S8 Normalise"] = ("PASS" if ok else "WARN",
                               f"{in_range*100:.2f}% pixels in [0,1]")
else:
    checks["S8 Normalise"] = ("SKIP", "No norm file")

# S9: clip
if CLIP_PATH and pathlib.Path(str(CLIP_PATH)).exists():
    with rasterio.open(str(CLIP_PATH)) as src:
        d   = src.read().astype("float32")
        crs = src.crs
        t   = src.transform
    ok_crs  = TARGET_CRS in str(crs)
    ok_data = d.std() > 0.001
    ok = ok_crs and ok_data
    checks["S9 Clip"] = ("PASS" if ok else "WARN",
                          f"CRS={crs}, pixel={t.a:.1f}m, shape={d.shape}")
else:
    checks["S9 Clip"] = ("SKIP", "No clip file")

# QGIS exports
qgis_files = list(QGIS_DIR.glob("*.tif"))
checks["QGIS Export"] = ("PASS" if len(qgis_files) >= 2 else "WARN",
                          f"{len(qgis_files)} GeoTIFF files + QML styles exported")

# Print report
STATUS_ICON = {
    "PASS": "✓", "PASS (recovered)": "✓ (GCP)", "WARN": "⚠", "FAIL": "✗", "SKIP": "–"
}
all_pass = True
for step, (status, detail) in checks.items():
    icon = STATUS_ICON.get(status, "?")
    if status in ("FAIL", "WARN"):
        all_pass = False
    print(f"  {icon}  {step:<20}  {status:<18}  {detail}")

print()
if all_pass:
    print("ALL CHECKS PASSED — ready to run nb_sar_09_accra_flood_intelligence_v2.ipynb")
else:
    print("Some checks failed or warned — review above before running main notebook.")

print()
print("QGIS files:")
for f in sorted(QGIS_DIR.glob("*")):
    print(f"  {f.name}")
print()
print(f"Output directory: {VAL_DIR.absolute()}")


In [ ]:
import pathlib

import pygeovision

pkg = pathlib.Path(pygeovision.__file__).parent
print("pygeovision package root:", pkg)
print()

# Find ALL files related to georeference and sar processing
for f in sorted(pkg.rglob("*.py")):
    name = str(f.relative_to(pkg))
    if any(kw in name.lower() for kw in ['georef', 'validator', 'sar', 'processor']):
        size = f.stat().st_size
        print(f"  {name:60s} ({size:,} bytes)")

In [ ]:
import pathlib

import pygeovision

pkg = pathlib.Path(pygeovision.__file__).parent
print("Package root:", pkg)
print()

# Show every sar and georeference related file with exact path
for f in sorted(pkg.rglob("*.py")):
    rel = str(f.relative_to(pkg))
    if any(kw in rel.lower() for kw in ['sar', 'georef', 'validator', 'processor']):
        print(f"{rel}  ({f.stat().st_size:,} bytes)")

print()
# Show what validate_sar_georeference actually imports from
sar_proc = pkg / "data" / "processors" / "sar.py"
if sar_proc.exists():
    src = sar_proc.read_text()
    # Find all imports
    for line in src.split('\n'):
        if 'import' in line and ('georef' in line.lower() or 'validator' in line.lower() or 'georeference' in line.lower()):
            print(f"IMPORT IN sar.py: {line}")
else:
    print(f"NOT FOUND: {sar_proc}")
    # Search for it
    for f in pkg.rglob("sar.py"):
        print(f"Found sar.py at: {f}")

In [ ]:
import pathlib

import pygeovision

pkg = pathlib.Path(pygeovision.__file__).parent

# Copy our fixed georeference.py to ALL locations where it might be imported from
import shutil

fixed_src = pathlib.Path("georeference.py")  # our fixed file we deployed earlier
if not fixed_src.exists():
    # Try to find it
    for loc in [
        pathlib.Path("./sar_validation"),
        pathlib.Path("."),
        pathlib.Path("/mnt/user-data/outputs"),
    ]:
        candidate = loc / "georeference.py"
        if candidate.exists():
            fixed_src = candidate
            break

print(f"Fixed source: {fixed_src}  exists={fixed_src.exists()}")

# Deploy to all georeference locations in the package
targets = list(pkg.rglob("georeference.py"))
print("\nExisting georeference.py locations:")
for t in targets:
    rel = str(t.relative_to(pkg))
    src_peek = t.read_text()[:200]
    has_gcp = 'from_gcps' in t.read_text()
    print(f"  {rel}  ({t.stat().st_size:,} bytes)  has_from_gcps={has_gcp}")

if fixed_src.exists():
    fixed_text = fixed_src.read_text()
    has_gcp = 'from_gcps' in fixed_text
    print(f"\nFixed file has from_gcps: {has_gcp}")

    if has_gcp:
        for t in targets:
            shutil.copy2(str(fixed_src), str(t))
            print(f"  Deployed to: {t.relative_to(pkg)}")
    else:
        print("Fixed file doesn't have GCP recovery — check path")
else:
    print("\nFixed georeference.py not found locally.")
    print("Download it from the outputs above and place it in the notebook directory.")

In [ ]:
from landsatxplore.api import API
from landsatxplore.hangar import Hangar

# 1. Define your USGS credentials
username = "YOUR_USGS_USERNAME"
password = "YOUR_USGS_PASSWORD"

# 2. Authenticate and initialize the API
api = API(username, password)

# 3. Search for available Landsat scenes
# Replace with your specific criteria (latitude, longitude, dates, dataset)
scenes = api.search(
    dataset='landsat_ot_c2_l2',  # Landsat 8-9 Collection 2 Level-2
    latitude=5.6037,             # e.g., Accra, Ghana
    longitude=-0.1870,
    start_date='2023-01-01',
    end_date='2023-12-31',
    max_cloud_cover=10
)

# Print results
print(f"Found {len(scenes)} scenes.")

if scenes:
    # Get the ID of the first scene found
    scene_id = scenes[0]['displayId']
    print(f"Downloading: {scene_id}")

    # 4. Download the scene
    # Hangar manages downloading and extracting the tar.gz product bundle
    downloader = Hangar(api, output_dir='./landsat_data')
    downloader.download(scene_id)
    print("Download completed!")

api.logout()
